# Temporal Distance of k-NN Neighbors

For each max-size EEG model x max-size vision model (VideoMAE-large, DINOv2-giant),
plots histograms of `|neighbor_window_index - anchor_window_index|` (windows apart),
aggregated over all W windows and all k=5 nearest neighbors.

Each subplot overlays:
- **Standard** cosine k-NN (solid bars)
- **Temporal-exclusion ±20 windows** (semi-transparent bars)

The **last section** investigates the LUNA x VideoMAE case, which shows unexplained
periodic peaks on top of the expected exponential decay in EEG neighbor distances.

In [ ]:
from __future__ import annotations
import os, io, base64, tarfile, functools
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.neighbors import NearestNeighbors
from scipy.signal import find_peaks
from scipy.ndimage import uniform_filter1d
from huggingface_hub import hf_hub_download

# ── Config ────────────────────────────────────────────────────────────────────
EMBED_REPO   = 'nitrox639/platonic-embeddings'
VIDEO_REPO   = 'Fudan-fMRI/CineBrain'
K            = 5
EXCL         = 20
MAX_CLIP_IDX = 2699
CLIP_SECONDS = 4.0
SEASON_END_S = (MAX_CLIP_IDX + 1) * CLIP_SECONDS   # 10 800 s
GIF_SIZE     = 160
SOURCE_FPS   = 8

for _tp in (Path('tokens/hf_token.txt'), Path('hf_token.txt'),
            Path.home() / 'hf_token.txt',
            Path('../tokens/hf_token.txt'),
            Path('../src/extraction_scripts/hf_token.txt')):
    if _tp.exists():
        os.environ['HF_TOKEN'] = _tp.read_text().strip()
        break
HF_TOKEN = os.environ.get('HF_TOKEN')
assert HF_TOKEN, 'No HF token found'

def _fetch(remote_path):
    local = hf_hub_download(EMBED_REPO, remote_path, repo_type='dataset', token=HF_TOKEN)
    return np.load(local, allow_pickle=True)['embeddings'].astype(np.float32)

# ── Model registries ──────────────────────────────────────────────────────────
_EEG = {
    'femba':       dict(max_size='large',  family='femba_luna', window_s=5,
                        path=lambda s: f'eeg/femba/femba_{s}_tusl_layerwise.npz'),
    'luna':        dict(max_size='huge',   family='femba_luna', window_s=5,
                        path=lambda s: f'eeg/luna/luna_{s}_layerwise.npz'),
    'neurolm':     dict(max_size='xl',     family='neurolm',    window_s=8,
                        path=lambda s: f'eeg/neurolm/neurolm_{s}_layerwise.npz'),
    'steegformer': dict(max_size='large',  family='steegformer', window_s=6,
                        path=lambda s: f'eeg/steegformer/steegformer_{s}_layerwise.npz'),
    'reve':        dict(max_size='large',  family='reve',        window_s=10,
                        path=lambda s: f'eeg/reve/reve_{s}_layerwise.npz'),
}
_VIS = {
    'videomae': dict(max_size='large',
                     path=lambda vs, fam: f'vision/videomae/videomae_{vs}__{fam}.npz'),
    'dinov2':   dict(max_size='giant',
                     path=lambda vs, fam: f'vision/dinov2/dinov2_{vs}__{fam}.npz'),
}
_LABEL = {'femba': 'FEMBA', 'luna': 'LUNA', 'neurolm': 'NeuroLM',
          'steegformer': 'STEEGFormer', 'reve': 'REVE',
          'videomae': 'VideoMAE', 'dinov2': 'DINOv2'}
_C = dict(eeg_std='#3a7eca', eeg_excl='#a8cce8',
          vis_std='#e07b39', vis_excl='#f0c09c')

# ── k-NN helpers ──────────────────────────────────────────────────────────────
def _l2(x):
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def _knn_std(z, k):
    nn = NearestNeighbors(n_neighbors=k + 1, metric='cosine', algorithm='brute').fit(z)
    return nn.kneighbors(z)[1][:, 1:].astype(np.int32)

def _knn_excl(z, k, excl):
    W = z.shape[0]
    D = (1.0 - z @ z.T).astype(np.float32)
    r = np.arange(W)
    D[np.abs(r[:, None] - r[None, :]) <= excl] = np.inf
    return np.argsort(D, axis=1)[:, :k].astype(np.int32)

def _layerwise_knn_eeg(emb, k, knn_fn):
    avg = emb.mean(axis=2)
    L = avg.shape[0]
    out = []
    for l in range(L):
        out.append(knn_fn(_l2(avg[l]), k))
        if (l + 1) % 8 == 0:
            print(f'    EEG layer {l+1}/{L}', flush=True)
    return np.stack(out)

def _layerwise_knn_vis(emb, k, knn_fn):
    if emb.ndim == 2: emb = emb[None]
    return np.stack([knn_fn(_l2(emb[l]), k) for l in range(emb.shape[0])])

def _mknn_score(ia, ib):
    return float((ia[:, :, None] == ib[:, None, :]).any(-1).mean())

# ── Caches ────────────────────────────────────────────────────────────────────
_eeg_cache = {}
_vis_cache = {}

def _get_eeg_knn(eeg_key, excl):
    key = (eeg_key, excl)
    if key not in _eeg_cache:
        cfg = _EEG[eeg_key]
        print(f'  EEG {_LABEL[eeg_key]}-{cfg["max_size"]} (excl={excl}) ...', flush=True)
        emb = _fetch(cfg['path'](cfg['max_size']))
        kfn = _knn_std if excl is None else (lambda z, k, e=excl: _knn_excl(z, k, e))
        _eeg_cache[key] = _layerwise_knn_eeg(emb, K, kfn)
    return _eeg_cache[key]

def _get_vis_knn(vis_key, family, excl):
    key = (vis_key, family, excl)
    if key not in _vis_cache:
        cfg = _VIS[vis_key]
        print(f'  Vision {_LABEL[vis_key]}-{cfg["max_size"]} (excl={excl}) ...', flush=True)
        emb = _fetch(cfg['path'](cfg['max_size'], family))
        kfn = _knn_std if excl is None else (lambda z, k, e=excl: _knn_excl(z, k, e))
        _vis_cache[key] = _layerwise_knn_vis(emb, K, kfn)
    return _vis_cache[key]

def analyze_pair(eeg_key, vis_key, excl=None):
    family = _EEG[eeg_key]['family']
    nn_e = _get_eeg_knn(eeg_key, excl)
    nn_v = _get_vis_knn(vis_key, family, excl)
    L_e, L_v = nn_e.shape[0], nn_v.shape[0]
    scores = np.array([[_mknn_score(nn_e[le], nn_v[lv])
                        for lv in range(L_v)] for le in range(L_e)])
    le, lv = np.unravel_index(scores.argmax(), scores.shape)
    tag = 'excl+{}'.format(excl) if excl else 'standard'
    print('  [{}] {}x{}: EEG-l{} x Vis-l{}  mKNN={:.4f}'.format(
        tag, eeg_key, vis_key, le, lv, scores[le, lv]))
    return dict(eeg_key=eeg_key, vis_key=vis_key,
                nn_eeg=nn_e[le], nn_vis=nn_v[lv],
                window_s=_EEG[eeg_key]['window_s'])

def temporal_dists(nn_idx):
    W = nn_idx.shape[0]
    return np.abs(nn_idx - np.arange(W)[:, None]).ravel()

print('Setup complete.')


In [ ]:
import cv2
from PIL import Image as PILImage
from IPython.display import HTML, display as ipy_display

CB_ROOT     = Path('.').resolve().parent
CLIP_CACHE  = CB_ROOT / '.clip_cache'
CLIP_MARKER = CLIP_CACHE / '.extracted'
_CLIP_INDEX = {}

def _build_clip_index(clips_dir):
    global _CLIP_INDEX
    _CLIP_INDEX.clear()
    for p in clips_dir.rglob('*.mp4'):
        digits = ''.join(c for c in p.stem if c.isdigit())
        if digits:
            _CLIP_INDEX[int(digits)] = p
    print(f'  Indexed {len(_CLIP_INDEX)} clips  '
          f'(range {min(_CLIP_INDEX)}..{max(_CLIP_INDEX)})')

def ensure_clips():
    if _CLIP_INDEX:
        return
    if CLIP_MARKER.exists():
        print(f'Using cached clips at {CLIP_CACHE}')
        _build_clip_index(CLIP_CACHE)
        return
    print('Downloading videos.tar (~2.6 GB) ...')
    from huggingface_hub import hf_hub_download as _dl
    tar_path = Path(_dl(repo_id=VIDEO_REPO, filename='videos.tar',
                        repo_type='dataset', token=HF_TOKEN))
    CLIP_CACHE.mkdir(parents=True, exist_ok=True)
    import tarfile as _tf
    with _tf.open(tar_path, 'r') as tar:
        tar.extractall(CLIP_CACHE)
    CLIP_MARKER.touch()
    _build_clip_index(CLIP_CACHE)

@functools.lru_cache(maxsize=512)
def _load_frames(clip_idx):
    cap = cv2.VideoCapture(str(_CLIP_INDEX[clip_idx]))
    frames = []
    while True:
        ok, f = cap.read()
        if not ok: break
        frames.append(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
    cap.release()
    return tuple(frames)

def get_window_frames(start_s, window_s, size=GIF_SIZE):
    end_s      = start_s + window_s
    first_clip = int(start_s // CLIP_SECONDS)
    last_clip  = min(int(np.ceil(end_s / CLIP_SECONDS)) - 1, MAX_CLIP_IDX)
    frames = []
    for c in range(first_clip, last_clip + 1):
        cf  = _load_frames(c)
        n   = len(cf)
        ct0 = c * CLIP_SECONDS
        f0  = max(0, int(np.floor((max(start_s, ct0) - ct0) / CLIP_SECONDS * n)))
        f1  = min(n, int(np.ceil((min(end_s, ct0 + CLIP_SECONDS) - ct0) / CLIP_SECONDS * n)))
        for frame in cf[f0:f1]:
            frames.append(cv2.resize(frame, (size, size), interpolation=cv2.INTER_LINEAR))
    return frames

def frames_to_gif(frames, fps=SOURCE_FPS):
    pil = [PILImage.fromarray(f) for f in frames]
    buf = io.BytesIO()
    pil[0].save(buf, format='GIF', save_all=True, append_images=pil[1:],
                duration=int(1000 / fps), loop=0, optimize=False)
    return buf.getvalue()

def gif_tag(frames, width=GIF_SIZE, border='none'):
    b64 = base64.b64encode(frames_to_gif(frames)).decode()
    return (f'<img src="data:image/gif;base64,{b64}" width="{width}" '
            f'style="border:{border}; image-rendering:auto;"/>')

print('Video helpers ready. Call ensure_clips() before visualizing.')


In [ ]:
def _draw_hist(ax, d_std, d_excl, c_std, c_excl, max_d):
    bins = np.arange(0, max_d + 2)
    ax.hist(d_std,  bins=bins, density=True, color=c_std,  alpha=0.85, label='standard')
    ax.hist(d_excl, bins=bins, density=True, color=c_excl, alpha=0.72,
            histtype='stepfilled', edgecolor='none', label=f'excl +/-{EXCL}')
    ax.axvline(EXCL, color='crimson', lw=0.9, ls='--', alpha=0.85)
    ax.set_xlim(0, max_d)
    ax.tick_params(labelsize=6)

def make_figure(pairs_std, pairs_excl, vis_label, save_as=None):
    n = len(pairs_std)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 5.2), squeeze=False)
    fig.subplots_adjust(hspace=0.5, wspace=0.35)
    for col, (ps, pe) in enumerate(zip(pairs_std, pairs_excl)):
        eeg_key = ps['eeg_key']
        W       = ps['nn_eeg'].shape[0]
        max_d   = W - 1
        d_eeg_s = temporal_dists(ps['nn_eeg'])
        d_vis_s = temporal_dists(ps['nn_vis'])
        d_eeg_e = temporal_dists(pe['nn_eeg'])
        d_vis_e = temporal_dists(pe['nn_vis'])
        ax = axes[0, col]
        _draw_hist(ax, d_eeg_s, d_eeg_e, _C['eeg_std'], _C['eeg_excl'], max_d)
        ax.set_title(f'{_LABEL[eeg_key]}-{_EEG[eeg_key]["max_size"]}', fontsize=8, pad=3)
        if col == 0:
            ax.set_ylabel('EEG density', fontsize=8)
        ax = axes[1, col]
        _draw_hist(ax, d_vis_s, d_vis_e, _C['vis_std'], _C['vis_excl'], max_d)
        ax.set_xlabel('windows apart', fontsize=7)
        if col == 0:
            ax.set_ylabel(f'{vis_label} density', fontsize=8)
    handles = [
        mpatches.Patch(fc=_C['eeg_std'],  label='EEG standard'),
        mpatches.Patch(fc=_C['eeg_excl'], label=f'EEG excl +/-{EXCL}'),
        mpatches.Patch(fc=_C['vis_std'],  label=f'{vis_label} standard'),
        mpatches.Patch(fc=_C['vis_excl'], label=f'{vis_label} excl +/-{EXCL}'),
        plt.Line2D([0],[0], color='crimson', lw=0.9, ls='--',
                   label=f'+/-{EXCL} boundary'),
    ]
    fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=7.5,
               bbox_to_anchor=(0.5, -0.03), frameon=True)
    fig.suptitle(
        f'EEG (top) x {vis_label} (bottom) -- k-NN neighbor temporal distance  k={K}',
        y=1.01, fontsize=10)
    if save_as:
        fig.savefig(save_as, dpi=150, bbox_inches='tight')
        print(f'Saved -> {save_as}')
    plt.show()

print('make_figure() ready.')


---
## EEG x VideoMAE

In [ ]:
eeg_models = list(_EEG.keys())

print('=== VideoMAE (standard) ===')
pairs_videomae_std  = [analyze_pair(m, 'videomae', excl=None) for m in eeg_models]

print('\n=== VideoMAE (excl +/-{}) ==='.format(EXCL))
pairs_videomae_excl = [analyze_pair(m, 'videomae', excl=EXCL)  for m in eeg_models]


In [ ]:
make_figure(pairs_videomae_std, pairs_videomae_excl,
            vis_label='VideoMAE', save_as='neighbor_dist_videomae.png')


---
## EEG x DINOv2

In [ ]:
print('=== DINOv2 (standard) ===')
pairs_dinov2_std  = [analyze_pair(m, 'dinov2', excl=None) for m in eeg_models]

print('\n=== DINOv2 (excl +/-{}) ==='.format(EXCL))
pairs_dinov2_excl = [analyze_pair(m, 'dinov2', excl=EXCL)  for m in eeg_models]


In [ ]:
make_figure(pairs_dinov2_std, pairs_dinov2_excl,
            vis_label='DINOv2', save_as='neighbor_dist_dinov2.png')


---
## LUNA x VideoMAE — Peak Analysis (standard k-NN, no exclusion)

The EEG distance histogram for LUNA shows **multiple peaks superposed on the
expected exponential decay**, visible as bumps at specific distances.

This section:
1. Detects those peak distances using a smooth-baseline residual approach.
2. Checks whether the same peaks appear in the VideoMAE distances (if so, the
   show itself has recurring visual patterns at those lags; if not, the
   phenomenon is EEG-specific).
3. Finds a concrete anchor window whose EEG nearest neighbours cluster on
   the peak distances, and displays the corresponding video clips.

In [ ]:
# ── Pull LUNA x VideoMAE standard results ────────────────────────────────────
luna_pair   = next(p for p in pairs_videomae_std if p['eeg_key'] == 'luna')
nn_eeg_luna = luna_pair['nn_eeg']   # (W, K) -- best EEG layer
nn_vis_luna = luna_pair['nn_vis']   # (W, K) -- best VideoMAE layer
W_luna      = nn_eeg_luna.shape[0]  # 2160
WIN_S_LUNA  = 5.0                   # seconds per window
starts_s    = np.arange(0.0, SEASON_END_S, WIN_S_LUNA)

dists_eeg = temporal_dists(nn_eeg_luna)
dists_vis = temporal_dists(nn_vis_luna)

# Full-range histogram: one bin per possible window-distance (0 .. W-1)
bins       = np.arange(0, W_luna + 1)
d_range    = np.arange(W_luna)
counts_eeg, _ = np.histogram(dists_eeg, bins=bins)
counts_vis, _ = np.histogram(dists_vis, bins=bins)

# ── Detect peaks in EEG via residual over a wide smooth baseline ─────────────
baseline   = uniform_filter1d(counts_eeg.astype(float), size=120)
residual   = counts_eeg.astype(float) - baseline

peaks_idx, _ = find_peaks(
    residual,
    prominence=residual.std() * 1.5,
    distance=20,
)
print(f'Detected {len(peaks_idx)} peak distances (windows): {peaks_idx.tolist()}')
if len(peaks_idx):
    peak_times_s   = peaks_idx * WIN_S_LUNA
    peak_times_min = peak_times_s / 60
    for p, ts, tm in zip(peaks_idx, peak_times_s, peak_times_min):
        print(f'  window {p:5d}  ->  {ts:6.0f} s  ({tm:.1f} min)')

# ── Plot: EEG distribution with detected peaks, + VideoMAE for comparison ────
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 7), sharex=True)

ax1.bar(d_range, counts_eeg, width=1, color=_C['eeg_std'], alpha=0.75, label='EEG counts')
ax1.plot(d_range, baseline, color='black', lw=1.2, ls='--', label='smooth baseline')
if len(peaks_idx):
    ax1.scatter(peaks_idx, counts_eeg[peaks_idx], color='red', zorder=5,
                s=40, label='detected peaks')
    for p in peaks_idx:
        ax1.axvline(p, color='red', lw=0.7, ls=':', alpha=0.5)
        ax1.annotate(
            f'{p}\n({p * WIN_S_LUNA / 60:.0f}m)',
            xy=(p, counts_eeg[p]),
            xytext=(p + 8, counts_eeg[p] * 1.05),
            fontsize=6, color='red', va='bottom')
ax1.set_ylabel('Neighbour pair count', fontsize=9)
ax1.legend(fontsize=8, loc='upper right')
ax1.set_title('LUNA-huge EEG k-NN: temporal distance distribution', fontsize=9)

ax2.bar(d_range, counts_vis, width=1, color=_C['vis_std'], alpha=0.75, label='VideoMAE counts')
if len(peaks_idx):
    for p in peaks_idx:
        ax2.axvline(p, color='red', lw=0.7, ls=':', alpha=0.5, label='EEG peak' if p == peaks_idx[0] else '')
ax2.set_ylabel('Neighbour pair count', fontsize=9)
ax2.set_xlabel('Windows apart  (1 window = 5 s)', fontsize=9)
ax2.legend(fontsize=8, loc='upper right')
ax2.set_title('VideoMAE-large k-NN: temporal distance distribution  (EEG peak positions shown in red)', fontsize=9)

fig.suptitle('LUNA-huge x VideoMAE-large -- full-range temporal distance histograms', fontsize=11)
plt.tight_layout()
plt.savefig('luna_videomae_peak_hist.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved luna_videomae_peak_hist.png')


In [ ]:
if not len(peaks_idx):
    print('No peaks detected -- adjust find_peaks parameters.')
else:
    peak_set = set(int(p) for p in peaks_idx)

    # For each anchor, count how many of its k EEG neighbours land on a peak distance
    peak_hits = np.array([
        sum(1 for j in range(K) if abs(int(nn_eeg_luna[w, j]) - w) in peak_set)
        for w in range(W_luna)
    ])
    max_hits = int(peak_hits.max())
    print(f'Max neighbours at peak distances per anchor: {max_hits}/{K}')

    candidates = np.where(peak_hits == max_hits)[0]
    # Prefer anchors away from boundaries
    interior = candidates[(candidates > 100) & (candidates < W_luna - 100)]
    anchor_w = int(interior[len(interior) // 2]) if len(interior) else int(candidates[0])

    neighbors   = nn_eeg_luna[anchor_w].astype(int)
    neigh_dists = np.abs(neighbors - anchor_w)
    at_peak     = [int(d) in peak_set for d in neigh_dists]

    t_anchor = float(starts_s[anchor_w])
    print(f'\nAnchor: window {anchor_w}  t = {t_anchor:.0f} s  '
          f'({int(t_anchor // 60)}m{int(t_anchor % 60):02d}s)')
    for j, (nb, d, peak) in enumerate(zip(neighbors, neigh_dists, at_peak)):
        marker = '  [PEAK]' if peak else ''
        t_nb   = float(starts_s[nb])
        print(f'  NN#{j+1}: window {nb:4d}  dist={d:4d}{marker}  '
              f't={t_nb:.0f}s ({int(t_nb//60)}m{int(t_nb%60):02d}s)')

    # ── Load video clips and show middle frame of each window ─────────────────
    ensure_clips()

    all_windows = [anchor_w] + neighbors.tolist()
    titles      = ['ANCHOR'] + [
        f'EEG-NN#{j+1}  d={int(d)}{" [PEAK]" if pk else ""}'
        for j, (d, pk) in enumerate(zip(neigh_dists, at_peak))
    ]
    colors = ['#e6a817'] + [
        '#c0392b' if pk else '#2a7ae2' for pk in at_peak
    ]

    n_panels = len(all_windows)
    fig, axes = plt.subplots(1, n_panels, figsize=(3.2 * n_panels, 3.8))
    if n_panels == 1: axes = [axes]

    for ax, win_idx, title, color in zip(axes, all_windows, titles, colors):
        t   = float(starts_s[win_idx])
        frm = get_window_frames(t, WIN_S_LUNA, size=GIF_SIZE)
        if frm:
            ax.imshow(frm[len(frm) // 2])
        else:
            ax.set_facecolor('#ccc')
        ax.set_title(title, fontsize=7.5, color=color, fontweight='bold', pad=3)
        ax.set_xlabel(
            f'w={win_idx}  {int(t // 60)}m{int(t % 60):02d}s',
            fontsize=7)
        for spine in ax.spines.values():
            spine.set_edgecolor(color)
            spine.set_linewidth(3)
        ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

    fig.suptitle(
        f'LUNA x VideoMAE -- anchor w={anchor_w} and its 5 EEG nearest neighbours\n'
        f'Red = neighbour at a detected peak distance, Blue = non-peak',
        fontsize=9)
    plt.tight_layout()
    plt.savefig('luna_videomae_peak_example.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved luna_videomae_peak_example.png')
